# Inference demo

## 1. Backbone output dimensions

In [1]:
import time
import torch
from omegaconf import OmegaConf
from dinov3.models import build_model_from_cfg

t0 = time.time()
cfg = OmegaConf.load("configs/pretrain.yaml")
model, embed_dim = build_model_from_cfg(cfg, only_teacher=True)
print(f"arch={cfg.student.arch} patch_size={cfg.student.patch_size} embed_dim={embed_dim}")

model = model.to_empty(device="cpu")
model.init_weights()
model.eval()
dummy = torch.rand(1, 1, 48, 48, 48) * 2 - 1
with torch.no_grad():
    out = model.forward_features(dummy)
print("clstoken", tuple(out["x_norm_clstoken"].shape))
print("patchtokens", tuple(out["x_norm_patchtokens"].shape))
print(f"elapsed={time.time() - t0:.2f}s")


arch=vit_large patch_size=16 embed_dim=1024


clstoken (1, 1024)
patchtokens (1, 27, 1024)
elapsed=2.03s


## 2. Features -> checkpoint -> logits

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GatedAttentionPooler(nn.Module):
    def __init__(self, M, L):
        super().__init__()
        self.V = nn.Sequential(nn.LayerNorm(M, eps=1e-6), nn.Linear(M, L), nn.Tanh())
        self.U = nn.Sequential(nn.LayerNorm(M, eps=1e-6), nn.Linear(M, L), nn.Sigmoid())
        self.w = nn.Linear(L, 1)

    def forward(self, H):
        A = self.w(self.V(H) * self.U(H)).transpose(1, 2)
        A = F.softmax(A, dim=2)
        return torch.bmm(A, H).squeeze(1)


class GateFusionClassifier(nn.Module):
    def __init__(self, emb_dim, hidden_dim, n_cls, dropout):
        super().__init__()
        self.attn = GatedAttentionPooler(M=emb_dim, L=hidden_dim)
        self.post = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Sequential(nn.Linear(emb_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, n_cls))

    def forward(self, x):
        z = self.attn(x)
        z = self.post(z)
        return self.head(z)


def load_classifier(ckpt_path, device="cpu"):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    state = ckpt["model"]
    if "config" in ckpt:
        arch = ckpt["config"]["architecture"]
        hidden_dim, dropout = arch["hidden_dim"], arch["dropout"]
        n_cls = len(next(iter(ckpt["config"]["tasks"].values()))["label_columns"])
    else:
        hidden_dim, dropout, n_cls = ckpt["hidden_dim"], ckpt["dropout"], ckpt["n_cls"]
    feat_dim = ckpt["feature_dim"]
    model = GateFusionClassifier(feat_dim, hidden_dim, n_cls, dropout)
    model.load_state_dict(state, strict=True)
    return model.eval().to(device)


In [3]:
import time
import numpy as np
import pandas as pd

t0 = time.time()
CKPT = "weights/classification_best.pt"
FEATS = "data/demo/features"
LABELS = "data/demo/labels.csv"

model = load_classifier(CKPT)
df = pd.read_csv(LABELS)
rows = []
with torch.no_grad():
    for _, r in df.iterrows():
        pid = str(r["PatientID"])
        x = np.load(f"{FEATS}/{pid}.npy").astype(np.float32)
        logits = model(torch.from_numpy(x)[None]).squeeze(0).numpy()
        rows.append({"PatientID": pid, "logit_0": float(logits[0]), "logit_1": float(logits[1])})
        print(pid, "logits", logits.round(4))
out = pd.DataFrame(rows)
out.to_csv("inference_results.csv", index=False)
elapsed = time.time() - t0
with open("run_info.txt", "w") as f:
    f.write(f"elapsed_seconds={elapsed:.2f}\nn_samples={len(out)}\n")
print(f"elapsed={elapsed:.2f}s  saved {len(out)} rows -> inference_results.csv, run_info.txt")


ISPY1_1008 logits [-9.0401  9.2941]
ISPY1_1011 logits [-10.4215  11.0872]
ISPY1_1017 logits [ 7.1976 -8.1531]
ISPY1_1029 logits [-8.4515  8.461 ]
ISPY1_1058 logits [-7.7998  9.3687]
ISPY1_1061 logits [-10.7123  11.1729]
ISPY1_1078 logits [ 7.1774 -8.1572]
ISPY1_1083 logits [ 7.6456 -8.3129]
ISPY1_1122 logits [ 7.882 -8.591]
ISPY1_1129 logits [ 7.2208 -8.2502]
ISPY1_1147 logits [ 7.9871 -9.2971]
ISPY1_1177 logits [-8.5554  8.7978]
ISPY1_1181 logits [-8.2361  9.5195]
ISPY1_1188 logits [ 8.1736 -8.9424]
ISPY1_1191 logits [-8.7267  8.8401]
ISPY1_1196 logits [-9.6136 10.0243]
ISPY1_1202 logits [-8.5211  8.754 ]
ISPY1_1218 logits [ 7.7728 -8.552 ]
ISPY1_1229 logits [ 7.2831 -8.2125]
ISPY1_1230 logits [ 7.7915 -8.7969]
elapsed=0.08s  saved 20 rows -> inference_results.csv, run_info.txt
